In [1]:
import pickle
import pandas as pd

print("Загрузка предсказаний\n")

# Загружаем предсказания
with open('data/processed/predictions_ensemble.pkl', 'rb') as f:
    predictions = pickle.load(f)

print(f"Загружено предсказаний для {len(predictions)} запросов")
print(f"\nПример первого запроса:")
first_query_id = list(predictions.keys())[0]
print(f"  query_id: {first_query_id}")
print(f"  Количество кандидатов: {len(predictions[first_query_id])}")
print(f"  Первые 5 item_id: {predictions[first_query_id][:5]}")

Загрузка предсказаний

Загружено предсказаний для 2452 запросов

Пример первого запроса:
  query_id: 70DfDUpwjxB4lzFd
  Количество кандидатов: 50
  Первые 5 item_id: ['4b566e71bcdbf903', 'd7ad9f91504e4d02', 'deb58d7ad8c27d06', '6042fecdd7753112', '9515d1e1ecdac72c']


In [2]:
print("\nСоздание answer.csv\n")

# Создаем DataFrame
query_ids = list(predictions.keys())
answers = [' '.join(predictions[qid]) for qid in query_ids]

answer_df = pd.DataFrame({
    'query_id': query_ids,
    'answer': answers
})

print(f"Создан DataFrame:")
print(f"  Shape: {answer_df.shape}")
print(f"  Колонки: {answer_df.columns.tolist()}")

# Сохраняем в CSV
answer_df.to_csv('answer.csv', index=False, encoding='utf-8')

print(f"\nайл answer.csv сохранен")
print(f"   Размер: {answer_df.shape[0]} строк")

# Проверяем размер файла
import os
file_size = os.path.getsize('answer.csv')
print(f"   Размер файла: {file_size / 1024:.1f} KB")


Создание answer.csv

Создан DataFrame:
  Shape: (2452, 2)
  Колонки: ['query_id', 'answer']

айл answer.csv сохранен
   Размер: 2452 строк
   Размер файла: 2078.5 KB


In [3]:
print("\nПроверка формата answer.csv\n")

# Загружаем сохраненный файл
answer_check = pd.read_csv('answer.csv')

print(f"1. Структура файла:")
print(f"   Shape: {answer_check.shape}")
print(f"   Колонки: {answer_check.columns.tolist()}")

# Проверяем количество строк
expected_queries = 2452
actual_queries = len(answer_check)
print(f"\n2. Количество запросов:")
print(f"   Ожидалось: {expected_queries}")
print(f"   Получено: {actual_queries}")
if expected_queries == actual_queries:
    print(f"   Правильное количество")
else:
    print(f"   Несоответствие! Нужно исправить")

# Проверяем количество item_id в каждом ответе
print(f"\n3. Проверка количества item_id:")
item_counts = answer_check['answer'].apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)
print(f"   Минимум: {item_counts.min()}")
print(f"   Максимум: {item_counts.max()}")
print(f"   Среднее: {item_counts.mean():.1f}")

if item_counts.max() <= 50:
    print(f"  Все ответы содержат не более 50 item_id")
else:
    print(f"  Есть ответы с более чем 50 item_id")

# Проверяем наличие всех query_id из benchmark
print(f"\n4. Проверка query_id:")
benchmark_queries = pd.read_parquet('data/raw/benchmark_queries.parquet')
benchmark_query_ids = set(benchmark_queries['query_id'].unique())
answer_query_ids = set(answer_check['query_id'].unique())

missing_queries = benchmark_query_ids - answer_query_ids
extra_queries = answer_query_ids - benchmark_query_ids

print(f"   Пропущенных запросов: {len(missing_queries)}")
print(f"   Лишних запросов: {len(extra_queries)}")

if len(missing_queries) == 0 and len(extra_queries) == 0:
    print(f"   Все query_id корректны")
else:
    if len(missing_queries) > 0:
        print(f"   Пропущенные: {list(missing_queries)[:3]}")
    if len(extra_queries) > 0:
        print(f"   Лишние: {list(extra_queries)[:3]}")

# Проверяем, что все item_id существуют в benchmark_items
print(f"\n5. Проверка item_id:")
benchmark_items = pd.read_parquet('data/raw/benchmark_items.parquet')
benchmark_item_ids = set(benchmark_items['item_id'].unique())

all_answer_items = set()
for answer in answer_check['answer']:
    if pd.notna(answer):
        all_answer_items.update(str(answer).split())

missing_items = all_answer_items - benchmark_item_ids
print(f"   Уникальных item_id в ответе: {len(all_answer_items)}")
print(f"   Отсутствующих в benchmark_items: {len(missing_items)}")

if len(missing_items) == 0:
    print(f"  Все item_id существуют в benchmark_items")
else:
    print(f"  Найдено {len(missing_items)} несуществующих item_id!")

# Проверяем повторы внутри строк
print(f"\n6. Проверка повторов:")
duplicates_found = 0
for idx, row in answer_check.iterrows():
    if pd.notna(row['answer']):
        items = str(row['answer']).split()
        if len(items) != len(set(items)):
            duplicates_found += 1

print(f"   Строк с повторами: {duplicates_found}")
if duplicates_found == 0:
    print(f"  Повторов нет")

if (expected_queries == actual_queries and 
    item_counts.max() <= 50 and 
    len(missing_queries) == 0 and 
    len(missing_items) == 0 and 
    duplicates_found == 0):
    print("ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ! Файл готов к отправке.")
else:
    print("Найдены проблемы. См. выше.")


Проверка формата answer.csv

1. Структура файла:
   Shape: (2452, 2)
   Колонки: ['query_id', 'answer']

2. Количество запросов:
   Ожидалось: 2452
   Получено: 2452
   Правильное количество

3. Проверка количества item_id:
   Минимум: 50
   Максимум: 50
   Среднее: 50.0
  Все ответы содержат не более 50 item_id

4. Проверка query_id:
   Пропущенных запросов: 0
   Лишних запросов: 0
   Все query_id корректны

5. Проверка item_id:
   Уникальных item_id в ответе: 66975
   Отсутствующих в benchmark_items: 0
  Все item_id существуют в benchmark_items

6. Проверка повторов:
   Строк с повторами: 0
  Повторов нет
ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ! Файл готов к отправке.


In [4]:
print("\nСодержимое answer.csv\n")

print("Первые 5 строк:")
print(answer_check.head())

print("\nПоследние 5 строк:")
print(answer_check.tail())

print(f"\nПример полного ответа для первого запроса:")
first_row = answer_check.iloc[0]
print(f"  query_id: {first_row['query_id']}")
print(f"  answer: {first_row['answer'][:100]}...")


Содержимое answer.csv

Первые 5 строк:
           query_id                                             answer
0  70DfDUpwjxB4lzFd  4b566e71bcdbf903 d7ad9f91504e4d02 deb58d7ad8c2...
1  JTrdTaZJvSiLPkXj  cbeccbecb1fb8d86 cf7cf3f80dbeeb5a f50963ef7dda...
2  LZCZNoVG4AFUkVRJ  d8fce513e4f000a7 3f89b8062dc85f1c 0b1c884e6c24...
3  660ac9QVtXkRxZC3  a846a2a5241e1180 4c15e7abba063ac5 3cce43c81145...
4  YgHcM9MVbxKnxD1e  fb0bf6077c59c5a2 56e3295572157d74 0c8a36e283d4...

Последние 5 строк:
              query_id                                             answer
2447  JYUbsR4EU5RtfGUA  db750e5083874ab3 854821d3c255d674 23d2ae1e90f2...
2448  SuCUWvKrKFRKTKuT  d6a506cb09afe336 003ca25f04a7eed1 fe9af5f098f6...
2449  fWIRLM5wCElVVjRV  082c39c8a5d4a9d4 98c8c98bbb109f15 2b396ad4c5b0...
2450  VjX4ayHpKHlftRgu  fa27c13c58a99573 59822dfd3e9ec127 6aef0c1ede43...
2451  lMQlWhgC1WeP0QFa  b067f1a1fc80c53a 84e1aa8b56c09e10 3c14eb495051...

Пример полного ответа для первого запроса:
  query_id: 70DfDUpwjxB4lz